# ACF and PACF

The autocorrelation function (ACF) measures linear association between a series and its lags; the partial autocorrelation function (PACF) removes the contribution of the intermediate lags. This notebook computes both by hand, compares them with `statsmodels`, and shows the textbook AR/MA identification patterns.

Companion note: [autocorrelation_function.md](../../notes/time_series/autocorrelation_function.md).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
plt.rcParams['figure.figsize'] = (8, 3.5)

def sample_acf(x, max_lag):
    z = np.asarray(x) - np.mean(x)
    denom = np.sum(z**2)
    return np.array([np.sum(z[:len(z)-h] * z[h:]) / denom for h in range(max_lag + 1)])

x = np.array([1., 2., 4., 3.])
print('hand-computed rho_hat(1) for (1,2,4,3):', sample_acf(x, 1)[1])

hand-computed rho_hat(1) for (1,2,4,3): 0.15


The note's worked example gives $\hat\gamma(0)=1.25$, $\hat\gamma(1)=0.1875$ and hence $\hat\rho(1)=0.15$. Next, simulate an AR(1) with $\phi=0.7$ and compare the sample ACF with $\rho(h)=0.7^h$.

In [2]:
n, phi = 2000, 0.7
e = rng.normal(size=n + 200)
x = np.zeros(n + 200)
for t in range(1, n + 200):
    x[t] = phi * x[t-1] + e[t]
x = x[200:]

from statsmodels.tsa.stattools import acf, pacf
lags = 12
r = sample_acf(x, lags)
print('max |hand - statsmodels|:', np.max(np.abs(r - acf(x, nlags=lags))))
fig, ax = plt.subplots()
ax.stem(range(lags + 1), r, label='sample ACF')
ax.plot(range(lags + 1), phi ** np.arange(lags + 1), 'ro--', label='theory 0.7^h')
ax.axhline(1.96 / np.sqrt(n), color='gray', ls=':'); ax.axhline(-1.96 / np.sqrt(n), color='gray', ls=':')
ax.set_xlabel('lag'); ax.legend(); plt.show()

max |hand - statsmodels|: 1.1102230246251565e-16


/tmp/ipykernel_78307/3663049007.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set_xlabel('lag'); ax.legend(); plt.show()


The sample ACF tails off geometrically along the theoretical curve. The dotted lines are the $\pm1.96/\sqrt n$ white-noise reference band.

In [3]:
def durbin_levinson(rho):
    """PACF from an ACF array rho[0..K] via the Durbin-Levinson recursion."""
    K = len(rho) - 1
    pac = np.zeros(K + 1); pac[0] = 1.0
    phi_prev = np.array([rho[1]]); pac[1] = rho[1]
    for k in range(2, K + 1):
        num = rho[k] - np.sum(phi_prev * rho[k-1:0:-1])
        den = 1 - np.sum(phi_prev * rho[1:k])
        pkk = num / den
        phi_new = np.append(phi_prev - pkk * phi_prev[::-1], pkk)
        phi_prev, pac[k] = phi_new, pkk
    return pac

p_hand = durbin_levinson(r)
p_sm = pacf(x, nlags=lags, method='ywm')
print('max |Durbin-Levinson - statsmodels yw|:', np.max(np.abs(p_hand - p_sm)))
print('PACF lags 1-4:', np.round(p_hand[1:5], 3))

max |Durbin-Levinson - statsmodels yw|: 7.632783294297951e-17
PACF lags 1-4: [ 0.717 -0.02  -0.001 -0.013]


For an AR(1) the PACF is about $0.7$ at lag 1 and close to zero afterwards. Now contrast an AR(2), an MA(1) and an ARMA(1,1), using exact theoretical ACF/PACF from `ArmaProcess`.

In [4]:
from statsmodels.tsa.arima_process import ArmaProcess
models = {
    'AR(2): phi=(0.75,-0.25)': ArmaProcess([1, -0.75, 0.25], [1]),
    'MA(1): theta=0.8': ArmaProcess([1], [1, 0.8]),
    'ARMA(1,1): phi=0.5, theta=0.5': ArmaProcess([1, -0.5], [1, 0.5]),
}
fig, axes = plt.subplots(len(models), 2, figsize=(9, 7))
for i, (name, proc) in enumerate(models.items()):
    axes[i, 0].stem(range(11), proc.acf(11)); axes[i, 0].set_title(name + ' ACF')
    axes[i, 1].stem(range(11), proc.pacf(11)); axes[i, 1].set_title('PACF')
plt.tight_layout(); plt.show()

/tmp/ipykernel_78307/4201223344.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


AR(2): ACF tails off, PACF cuts off after lag 2. MA(1): ACF cuts off after lag 1, PACF tails off. ARMA(1,1): both tail off. A trend breaks these rules, as the last cell shows: a random walk has a slowly decaying ACF even though no stationary AR mechanism is involved.

In [5]:
rw = np.cumsum(rng.normal(size=500))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(rw); axes[0].set_title('random walk')
axes[1].stem(acf(rw, nlags=30)); axes[1].set_title('ACF of random walk')
plt.tight_layout(); plt.show()
print('ACF of first difference at lag 1:', round(acf(np.diff(rw), nlags=1)[1], 3))

ACF of first difference at lag 1: 0.044


/tmp/ipykernel_78307/2060093544.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()
